# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis (grain):** one row = one `client_hash_id` + `content_hash_id` + `report_date`.  
This is the native grain of `fact_content_daily_performance` (the table contains no keyword or URL dimensions). My modeling aggregates over this grain by summing `gsc_impressions` and `gsc_clicks` and averaging `gsc_avg_position` for each content item over a window of days.

**Table(s) used:** `fact_content_daily_performance` (partitioned by `month=YYYY-MM`) as the primary source. `dim_content` is joined for static attributes (content creation date, word count). `dim_clients` is used only to check history depth. All joins are read‑only and never aggregated.

**Time window:** development month is `2026-03`. The final month (`2026-06`) is **sealed** and used only for query tests, never to shape labels.

**What I'd predict/rank (label or proxy):** `ctr_gap = tier_avg_ctr - ctr` (proxy for under‑performance relative to position tier). Computed from `gsc_clicks / gsc_impressions` over a trailing window (first 15 days of March). Positive `ctr_gap` means the page earns fewer clicks than peers at the same position tier.

**Excluded:** GA4 columns (because `ga4_data_available` is three‑valued and often zero‑filled) and AI‑referral columns (too sparse to support stable tier‑level comparisons).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError("HF_TOKEN environment variable not set.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FILTERED = f"(SELECT * FROM {MARCH} WHERE gsc_data_available IS TRUE AND gsc_impressions > 0)"

grain_probe = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {FILTERED}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Grain probe on March 2026 data – expecting 0 rows (unique on the three keys).")
print(grain_probe)
print(f"Rows returned: {len(grain_probe)}")

Grain probe on March 2026 data – expecting 0 rows (unique on the three keys).
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []
Rows returned: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (knowable at the decision moment):
- `gsc_impressions` – daily impressions
- `gsc_avg_position` – daily average ranking position
- `position_tier` (derived from `gsc_avg_position` via binning)
- `gsc_clicks` (trailing window, not the target day)
- `gsc_data_available` – filter flag

**Label / proxy:** `ctr_gap = tier_avg_ctr - ctr` (positive = under‑performing).

**Context:** `client_hash_id`, `content_hash_id`, `report_date` – used for grouping/joining, never modeled.

**Excluded:** GA4 columns (zero‑fill risk), AI‑referral columns (sparse).

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FILTERED}
""").df()
print("Slice size and date span, March 2026 (filtered to GSC available + impressions > 0)")
print(slice_check)

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_impressions > 0 AND gsc_data_available IS TRUE) AS rows_with_real_impressions
    FROM {MARCH}
""").df()
print("Availability check, IS TRUE filter")
print(availability_check)
survive_pct = availability_check["gsc_available_rows"][0] / availability_check["total_rows"][0] * 100
print(f"{survive_pct:.1f} percent of March rows have gsc_data_available = TRUE")

Slice size and date span, March 2026 (filtered to GSC available + impressions > 0)
   total_rows  n_clients  n_content_items   min_date   max_date
0     3611061         47           176738 2026-03-01 2026-03-31
Availability check, IS TRUE filter
   total_rows  gsc_available_rows  rows_with_real_impressions
0     9841378             3611061                     3611061
36.7 percent of March rows have gsc_data_available = TRUE


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. **`gsc_impressions_prev`** – summed over first 15 days, knowable before scoring.
2. **`gsc_avg_position_prev`** – averaged over first 15 days.
3. **`position_tier`** – binned from `avg_position_prev`; static for the window.
4. **`ctr_prev`** – `clicks_prev / impressions_prev` from the same trailing window.
5. **`has_min_volume`** – `impressions_prev >= 100`.

All are computed exclusively from past data (days 1‑15), so no leakage.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
prev = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_prev,
        SUM(gsc_clicks) AS gsc_clicks_prev,
        AVG(gsc_avg_position) AS gsc_avg_position_prev
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date < DATE '2026-03-16'
    GROUP BY client_hash_id, content_hash_id
""").df()

target_day = con.sql(f"""
    SELECT client_hash_id, content_hash_id, gsc_clicks, gsc_impressions
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date = DATE '2026-03-31'
""").df()

prev["ctr_prev"] = prev["gsc_clicks_prev"] / prev["gsc_impressions_prev"].replace(0, np.nan)
prev["has_min_volume"] = (prev["gsc_impressions_prev"] >= 100).astype(int)

bins = [0, 3, 10, 20, 50, np.inf]
labels_ = ["top_3", "page_1", "striking", "page_3_5", "deep"]
prev["position_tier"] = pd.cut(prev["gsc_avg_position_prev"], bins=bins, labels=labels_)

feat = prev[prev["has_min_volume"] == 1].dropna(subset=["ctr_prev", "position_tier"]).copy()
feat["tier_avg_ctr_prev"] = feat.groupby("position_tier", observed=True)["ctr_prev"].transform("mean")

target_day["ctr_31"] = target_day["gsc_clicks"] / target_day["gsc_impressions"].replace(0, np.nan)
labeled = feat.merge(
    target_day[["client_hash_id", "content_hash_id", "ctr_31", "gsc_clicks"]],
    on=["client_hash_id", "content_hash_id"], how="inner"
).dropna(subset=["ctr_31"])

labeled["ctr_gap"] = labeled["tier_avg_ctr_prev"] - labeled["ctr_31"]
labeled["is_underperformer"] = (labeled["ctr_gap"] > 0).astype(int)

FEATURES = ["gsc_impressions_prev", "gsc_avg_position_prev", "position_tier", "ctr_prev", "has_min_volume"]
print(f"Five feature frame: {feat.shape[0]} rows, {len(FEATURES)} features")
print(feat[FEATURES].head())

from sklearn.tree import DecisionTreeRegressor

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X = pd.get_dummies(labeled[FEATURES], columns=["position_tier"])
y = labeled["is_underperformer"]

honest_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X, y)
honest_score = honest_model.predict(X)
print(f"Honest Precision@50, 5 features: {precision_at_k(honest_score, y, 50):.3f}")

X_leaky = X.copy()
X_leaky["TRAP_target_day_clicks"] = labeled["gsc_clicks"].values
leaky_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_score = leaky_model.predict(X_leaky)
print(f"Leaky Precision@50, target day clicks added: {precision_at_k(leaky_score, y, 50):.3f}")
del X_leaky
print(f"Trap removed, kept honest number: Precision@50 = {precision_at_k(honest_score, y, 50):.3f}")

Five feature frame: 77540 rows, 5 features
    gsc_impressions_prev  gsc_avg_position_prev position_tier  ctr_prev  \
0                  296.0               2.168919         top_3  0.000000   
1                  941.0               2.172077         top_3  0.002125   
8                  859.0               2.407289         top_3  0.001164   
9                  391.0              10.656093      striking  0.000000   
10                1153.0               2.705555         top_3  0.004337   

    has_min_volume  
0                1  
1                1  
8                1  
9                1  
10               1  
Honest Precision@50, 5 features: 0.960
Leaky Precision@50, target day clicks added: 1.000
Trap removed, kept honest number: Precision@50 = 0.960


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Unbalanced panel + tier volume floors.** Clients have different start dates; within March, `position_tier` strata are uneven – `top_3` has higher median impressions than `striking`. Even after `has_min_volume`, low‑volume tiers have noisier CTR gaps. This slice cannot tell us why a gap exists (title, SERP features, intent) – only that it exists.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tier_volume = feat.groupby("position_tier", observed=True)["gsc_impressions_prev"].median().sort_values(ascending=False)
print("Median trailing impressions by position_tier, March, has_min_volume rows")
print(tier_volume)

client_history = con.sql(f"""
    SELECT gsc_data_start, COUNT(*) AS n_clients
    FROM read_parquet('{REL}/dim_clients.parquet')
    WHERE gsc_data_start IS NOT NULL
    GROUP BY gsc_data_start
    ORDER BY gsc_data_start
    LIMIT 5
""").df()
print("Earliest client gsc_data_start values, panel is unbalanced")
print(client_history)

Median trailing impressions by position_tier, March, has_min_volume rows
position_tier
top_3       896.0
page_1      619.0
page_3_5    565.0
striking    425.0
deep        183.5
Name: gsc_impressions_prev, dtype: float64
Earliest client gsc_data_start values, panel is unbalanced
  gsc_data_start  n_clients
0     2025-01-27          2
1     2025-02-11          1
2     2025-03-11          1
3     2025-06-07          1
4     2025-06-18          1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.